# Blackboard

https://arxiv.org/pdf/2507.01701

## Board

In [1]:
from pydantic import BaseModel, Field, model_validator
from enum import Enum
import uuid

def get_id(n : int = 6):
    uuid4 = uuid.uuid4().hex[:n]
    return str(uuid4).replace('-', '')

class NoteStatus(Enum):
    ACTIVE='active'
    """Запись активна"""
    MARKED_FOR_DELETION='marked_for_deletion'
    """Помечена к удалению"""
    DELETED='deleted'
    """Заметка была удалена"""

class BaseNote(BaseModel):
    content : str = Field(max_length=2000, description='Содержимое записи')
    summary : str = Field(max_length=280, description='Суть записи в одном предложении')
    keywords : list[str] = Field(min_length=1, max_length=5, description='Ключевые слова')

class Note(BaseNote):
    id : str = Field(default='', description='ID записи')
    # status : NoteStatus = Field(default=NoteStatus.ACTIVE, description='Статус записи')
    author_id : str = Field(description='ID автора записи')

    @model_validator(mode='after')
    def _validate_model(self):
        self.id = get_id()
        return self

class Board(BaseModel):
    question : str = Field(description='Вопрос пользователя')
    notes : list[BaseNote] = Field(default=[], description='Список записей')

    def get_notes(self, last_n : int | None = None) -> list[dict]:
        """
        Возвращает список актуальных записей на доске с краткой информацией.
        Если передан last_n, вернет последние last_n записей.
        """
        notes = [{
            'id': note.id,
            'summary': note.summary,
            'keywords': note.keywords
        } for note in self.notes]
        if last_n is not None:
            notes = notes[-last_n:]
        return notes
    
    def get_note(self, note_id : str) -> Note | None:
        """
        Возвращает запись с указанным id.
        Возвращает None, если запись не найдена.
        """
        notes = [n for n in self.notes if n.id == note_id]
        if len(notes) == 0:
            return None
        return notes[0]
    
    def add_note(self, note : BaseNote, author_id : str) -> str:
        """
        Добавляет запись на доску.
        Возвращает id записи.
        """
        note = Note(author_id=author_id, **note.model_dump())
        self.notes.append(note)
        return note.id
    
    def remove_note(self, note_id : str):
        self.notes = [n for n in self.notes if n.id != note_id]

## Agents

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from src import llm

In [3]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import BaseMessage

STRUCTURED_RESPONSE_KEY = 'structured_response'
MESSAGES_KEY = 'messages'

class Agent():

    def __init__(
        self,
        *args,
        id_ : str | None = None,
        tools : list | None = None,
        system_prompt : str | None = None,
        response_format : type[BaseModel] | None = None,
        checkpointer : InMemorySaver | None = None,
        summarization_tokens : int = 4000, 
        summarization_keep : int = 10,
        **kwargs
    ):
        self.id = id_ or get_id()
        
        self.tools = tools or []
        self.system_prompt = self._format_system_prompt(system_prompt)
        self.response_format = response_format
        self.checkpointer = checkpointer or InMemorySaver()

        summarization_middleware = SummarizationMiddleware(
            model=llm,
            trigger=("tokens", summarization_tokens),
            keep=("messages", summarization_keep)
        )
        
        self._agent = create_agent(
            *args,
            model=llm, 
            tools=tools, 
            response_format=response_format, 
            system_prompt=self.system_prompt,
            checkpointer=self.checkpointer,
            middleware=[summarization_middleware],
            **kwargs
        )

    def _format_system_prompt(self, system_prompt : str | None) -> str | None:
        return system_prompt

    def invoke(self, messages : BaseMessage, thread_id : str | None = None, force : bool = False):
        response = None

        def invoke():
            return self._agent.invoke(
                input={'messages': messages},
                config={"configurable": {"thread_id": thread_id or self.id}},
            )

        if force:
            while response is None:
                try:
                    response = invoke()
                except:
                    print('Maybe tool calls or something idk')
        else:
            response = invoke()
            
        if STRUCTURED_RESPONSE_KEY in response:
            return response[STRUCTURED_RESPONSE_KEY]
        return response[MESSAGES_KEY][-1].content
    
    @property
    def info(self):
        return {
            'id': self.id
        }
    
class Role(BaseModel):
    name : str = Field(description='Название роли')
    description : str = Field(max_length=140, description='Описание роли')
    
class RoleAgent(Agent):

    def __init__(self, role : Role, *args, **kwargs):
        self.role = role
        super().__init__(*args, **kwargs)

    def _format_system_prompt(self, system_prompt):
        if system_prompt is None:
            return system_prompt
        return system_prompt.format(**self.info)

    @property
    def info(self):
        return {
            **super().info,
            'role_name': self.role.name,
            'role_description': self.role.description
        }

### Generator

In [4]:
class GeneratorResponse(BaseModel):
    roles : list[Role] = Field(min_length=1, max_length=3, description='Список экспертных ролей')

_generator_prompt = """
Вам дан вопрос. Предоставьте мне список экспертных ролей, которые наиболее полезны для решения вопроса.
Вопрос:
{question}
"""

def create_generator_agent(board : Board) -> Agent:
    system_prompt = _generator_prompt.format(question=board.question)
    return Agent(system_prompt=system_prompt, response_format=GeneratorResponse)

### Expert

In [5]:
_id_prompt = """
Ваш личный id {id}.
"""
_system_prompt = """
Вы {role_name}, сотрудничающий с другими агентами для решения задачи.
Задача:
{question}
Существует общая доска, которую каждый из вас может читать и на которой может оставлять записи.
"""
_expert_prompt = """
Вы выдающийся специалист:
{role_name}
Описание:
{role_description}

Основываясь на ваших экспертных знаниях и текущем содержимом общей доски, решите задачу, изложите свои идеи и информацию, которую вы хотите записать на доску.
Совершенно не обязательно полностью соглашаться с точкой зрения, представленной на доске.
"""

def create_expert_agent(role : Role, board : Board, ):
    system_prompt = _id_prompt + (_system_prompt + _expert_prompt).format(
        question=board.question,
        role_name=role.name,
        role_description=role.description
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_notes),
            tool(board.get_note)  
        ],
        response_format=BaseNote,
    )
    return agent

### Decider

In [6]:
class DeciderResponse(BaseModel):
    note : BaseNote = Field(description='Запись для добавления на доску')
    final_answer : str | None = Field(default=None, description='Окончательный развернутый ответ на задачу')

_decider_prompt = """
Ваша задача проанализировать текущее состояние общей доски и решить, достаточно ли у команды информации для получения окончательного ответа.
Если информации на доске достаточно для решения задачи, вы должны указать, что работа завершена, и предоставить окончательный ответ.
Если для получения ответа необходима дополнительная информация от других агентов, вы должны указать, что процесс следует продолжить.
"""

def create_decider_agent(board : Board):
    role = Role(
        name='Арбитр',
        description='Оценивает полноту информации. Выдает финальный ответ, либо инициирует продолжение обсуждения'
    )
    system_prompt = _id_prompt + (_system_prompt + _decider_prompt).format(
        role_name=role.name,
        question=board.question,
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_notes),
            tool(board.get_note)  
        ],
        response_format=DeciderResponse,
    )
    return agent

### Planner

In [7]:
_planner_prompt = """
Создайте план решения исходной задачи на основе текущего содержимого общей доски.
Опишите задачу своими словами, затем изложите пошаговый план её решения.
Если план уже существует на доске или задача достаточно проста для прямого решения, просто укажите, что нет необходимости декомпозировать задачи и вы ожидаете дополнительной информации.
Не решайте задачу. Предоставьте только план.
"""

def create_planner_agent(board : Board):
    role = Role(
        name='Планировщик',
        description='Разрабатывает пошаговый план решения задачи на основе содержимого доски'
    )
    system_prompt = _id_prompt + (_system_prompt + _planner_prompt).format(
        role_name=role.name,
        question=board.question,
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_notes),
            tool(board.get_note)  
        ],
        response_format=BaseNote,
    )
    return agent

### Critic

In [8]:
_critic_prompt = """
Проанализируйте записи на общей доске и определите любые бесполезные или избыточные.

Если вы обнаружите такие записи, опишите их и объясните, почему они должны быть удалены.
Если бесполезных записей нет, просто укажите, что бесполезных записей нет и вы ожидаете дополнительной информации.
"""

def create_critic_agent(board : Board):
    role = Role(
        name='Критик',
        description='Выявляет ошибочные или вводящие в заблуждение записи на доске'
    )
    system_prompt = _id_prompt + (_system_prompt + _critic_prompt).format(
        role_name=role.name,
        question=board.question,
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_notes),
            tool(board.get_note)  
        ],
        response_format=BaseNote,
    )
    return agent

### Cleaner

In [9]:
_cleaner_prompt = """
Проанализируйте записи на общей доске и определите любые бесполезные или избыточные.

Если вы обнаружите такие записи:
- Перечислите их
- Для каждого объясните, почему оно бесполезно или избыточно

Если бесполезных записей нет, просто укажите, что бесполезных записей нет и вы ожидаете дополнительной информации.
"""

class CleanerResponse(BaseModel):
    note : BaseNote = Field(description='Запись для добавления на доску')
    notes_ids : list[str] = Field(default=[], description='Список ID записей к удалению')

def create_cleaner_agent(board : Board):
    role = Role(
        name='Уборщик',
        description='Анализирует доску, выявляет и удаляет бесполезные или избыточные записи'
    )
    system_prompt = _id_prompt + (_system_prompt + _cleaner_prompt).format(
        role_name=role.name,
        question=board.question,
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_notes),
            tool(board.get_note)  
        ],
        response_format=CleanerResponse,
    )
    return agent

### Controller

In [10]:
class ControllerResponse(BaseModel):
    agents_ids : list[str] = Field(min_length=1, description='Упорядоченный список ID агентов')

_controller_prompt = """
Ваша задача назначить других агентов для сотрудничества и решения данной задачи.
Имена агентов и их описания перечислены ниже:
{agents}
Данная задача:
{question}
Агенты обмениваются информацией через общую доску.
Основываясь на содержимом, которое уже есть на доске, вам необходимо выбрать подходящих агентов из списка, чтобы они оставили записи на доске.
"""

def create_controller_agent(role_agents : list[RoleAgent], board : Board):
    system_prompt = _controller_prompt.format(
        agents=[rl.info for rl in role_agents],
        question=board.question
    )
    return Agent(
        system_prompt=system_prompt,
        tools=[
            tool(board.get_notes),
            tool(board.get_note)  
        ],
        response_format=ControllerResponse,
    )

In [11]:
def _get_agent(agent_id : str, agents : list[Agent]) -> Agent | None:
    for agent in agents:
        if agent.id == agent_id:
            return agent
    return None

def get_agents(agents_ids : list[str], agents : list[Agent]):
    result = []
    for agent_id in agents_ids:
        agent = _get_agent(agent_id, agents)
        if agent is not None:
            result.append(agent)
    return result

## Experiment

In [12]:
question = """
Городская идентичность г. Тюмени
"""

board = Board(question=question)

generator_agent = create_generator_agent(board)
roles = generator_agent.invoke([]).roles

In [13]:
expert_agents = [create_expert_agent(r, board) for r in roles]

for expert in expert_agents:
    print(expert.role.name, expert.role.description)

Градостроительный историк Исследует историческое развитие Тюмени, её архитектурные стили, планировку и трансформацию городской среды с момента основания до наших дней
Культурный антрополог Анализирует традиции, обычаи, символы и коллективную память жителей, формирующие уникальную культурную идентичность города
Социолог‑урбанист Изучает социальные структуры, демографию, миграционные потоки и их влияние на формирование городской идентичности


In [ ]:
decider_agent = create_decider_agent(board)
planner_agent = create_planner_agent(board)
critic_agent = create_critic_agent(board)
cleaner_agent = create_cleaner_agent(board)

role_agents = [planner_agent, *expert_agents, critic_agent, cleaner_agent, decider_agent]
controller_agent = create_controller_agent(role_agents, board)

In [15]:
from tqdm import tqdm

final_answer = None

for i in range(10):
    print(f'Итерация {i+1}')
    agents_ids = controller_agent.invoke(
        [f'Записей на доске: {len(board.notes)}'], force=True
    ).agents_ids
    agents = get_agents(agents_ids, role_agents)
    print(str.join('\n', ['- ' + a.role.name for a in agents]))

    for a in agents:

        response = a.invoke([f'Записей на доске: {len(board.notes)}'], force=True)
        if a == decider_agent:
            final_answer = response.final_answer
            if final_answer is not None:
                break
            note = response.note
        elif a == cleaner_agent:
            notes_ids = response.notes_ids
            for note_id in notes_ids:
                board.remove_note(note_id)
            note = response.note
        else:
            note = response
        note_id = board.add_note(note, a.id)
        print(f'{note_id}, {a.role.name}: {note.summary}')

    if final_answer is not None:
        break

Итерация 1
- Градостроительный историк
- Культурный антрополог
- Социолог‑урбанист
- Планировщик
c23b92, Градостроительный историк: Градостроительный обзор идентичности Тюмени: исторический контекст, основные архитектурные стили, их влияние на образ города и рекомендации по дальнейшему развитию.
5a05a6, Культурный антрополог: Культурно‑антропологический обзор идентичности Тюмени: коллективная память, символика, традиции и их роль в формировании уникального образа города.
d4dc4c, Социолог‑урбанист: Социолог‑урбанистический обзор идентичности Тюмени: демография, миграция, социальные структуры, пространственная сегрегация, рекомендации по интеграции и формированию инклюзивного городского нарратива.


Deserializing unregistered type __main__.ControllerResponse from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ControllerResponse')]


c60a79, Планировщик: Plan for solving the task of defining Tyumen's city identity, integrating existing notes on urban, cultural, sociological aspects.
Итерация 2
- Критик
- Арбитр
- Уборщик
f7dc40, Критик: No useless entries found; awaiting further information.
8b8b3f, Арбитр: Синтезированный аналитический материал, объединяющий градостроительные, культурные и социологические аспекты, раскрывающий городскую идентичность Тюмени и предлагающий рекомендации по её укреплению.


Deserializing unregistered type __main__.ControllerResponse from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ControllerResponse')]


c7184f, Уборщик: На доске обнаружены две записи, которые не добавляют новой информации к задаче и могут быть удалены.
Итерация 3


...

13b7c9, Культурный антрополог: Интентный (нео‑материальный) портрет Тюмени: праздники, язык, кулинарные символы и механизмы их сохранения, формирующие живую городскую идентичность.


Deserializing unregistered type __main__.BaseNote from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'BaseNote')]


c0a6e1, Социолог‑урбанист: Социолог‑урбанистический вклад: роль цифровых сообществ, миграционных субкультур, публичных ритуалов в формировании городской идентичности Тюмени; рекомендации по созданию интеграционных хабов и поддержке гибридных культурных практик.


Deserializing unregistered type __main__.DeciderResponse from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'DeciderResponse')]


483305, Критик: Only note c7184f is redundant and should be removed.


In [19]:
print(final_answer)

### Городская идентичность Тюмени

**Тюмень** – это город‑перекрёсток, где **историческое наследие**, **культурные практики** и **социальные динамики** тесно переплетаются, формируя уникальный образ:

1. **Исторический фундамент** – крепость 1586 г., центр торговли Сибирью, классические и модернистские архитектурные слои, советская плановая застройка и пост‑советские проекты. Всё это создает визуальный «слоистый» ландшафт, где старое соседствует с новым.
2. **Коллективная память и символика** – крепостные стены, река Тобол, Казённый двор, ежегодные фестивали «Сибирский путь», День основания, памятные доски у заводов. Эти символы поддерживают чувство принадлежности и исторической преемственности.
3. **Социальный портрет** – население ~800 тыс., молодая трудоспособная часть (≈½), мультиэтнический состав (русские ≈ 85 %, татары ≈ 5 %, остальные ≈ 10 %). Миграционные потоки из регионов России и стран СНГ приносят новые субкультуры, усиливая креативный потенциал.
4. **Современные практики**

In [20]:
print(note.content)

После анализа всех 8 записей на общей доске выявлена **одна явно избыточная запись**:

1. **Запись `c7184f`** – «На доске обнаружены две записи, которые не добавляют новой информации к задаче и могут быть удалены». Эта запись сама по себе не содержит исследовательского материала, а лишь мета‑комментарий о предполагаемой избыточности других записей. Она не вносит никакой ценности в процесс определения городской идентичности Тюмени и, более того, не уточняет, какие именно две записи следует удалить. Поэтому её присутствие лишь загромождает доску и может вводить в заблуждение.

**Почему остальные записи полезны и не являются избыточными**

| ID | Краткое содержание | Почему сохраняется |
|----|--------------------|--------------------|
| `c23b92` | Градостроительный обзор (история, архитектурные стили, рекомендации). | Предоставляет фундаментальный историко‑архитектурный контекст. |
| `5a05a6` | Культурно‑антропологический обзор (коллективная память, символика, традиции). | Охватывает нем

In [21]:
board.notes

[Note(content='### Градостроительный обзор идентичности Тюмени\n\n**1. Краткая историческая справка**\n- **Основание (1586\u202fг.)** – крепость Тюмень возникла как фортификационный пункт на границе Сибирского пути. Первоначальная планировка была типична для русских крепостей: прямоугольный план, деревянные стены, центральный **Казённый двор** (позже — площадь Тюмени) и церковный комплекс (Тюменский собор, 1662\u202fг.).\n- **XV‑XIX вв.** – развитие как торговый центр Сибирского пути. Появление деревянных и кирпичных жилых кварталов, улиц‑трасс, соединяющих центр с реками Тобол и Тюмень. Архитектура в стиле **русского классицизма** (домики с резными наличниками, фасады из кирпича, лепнина).\n- **Конец XIX – начало XX\u202fв.** – индустриализация, железная дорога (1884\u202fг.) и рост населения. Появление **модернистских** и **ар-нуво** построек (дом «Караул», 1905\u202fг.), а также первых многоквартирных домов в стиле **неоклассицизма**.\n- **Советский период (1920‑1991)** – плановая з